# 1. Importar librerías

Preparación de las herramientas para revisar datos, crear transformaciones y comprobar sus resultados.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 2. Carga del dataset

Se carga el CSV original del repositorio del grupo, sin modificar los datos de origen.

In [2]:
# Carga del dataset desde GitHub

url = "https://raw.githubusercontent.com/sofkp/Machine-Learning-Project/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dimensiones del dataset:", df.shape)

df.head()

Dimensiones del dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 3. Revisión de datos

Se transforma `TotalCharges` a numérico y se verifica la calidad de los registros. **La única variable con faltantes es `TotalCharges`**; todos esos registros tienen `tenure = 0`.

In [3]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Valores faltantes por columna:")

print(df.isna().sum()[df.isna().sum() > 0])

print("\nFilas duplicadas:", df.duplicated().sum())

print("IDs duplicados:", df["customerID"].duplicated().sum())


# Verificar que todos los TotalCharges faltantes
# correspondan a clientes con tenure = 0

assert (
    df.loc[df["TotalCharges"].isna(), "tenure"] == 0
).all(), "Se encontraron faltantes con tenure distinto de 0"

print("Verificado: todos los faltantes tienen tenure = 0")

# Las demás características están completas en este CSV estático.
columnas_sin_imputacion = [col for col in df.columns
                           if col not in ("TotalCharges",)]
assert not df[columnas_sin_imputacion].isna().any().any(), (
    "Se encontraron faltantes fuera de TotalCharges; revisar el CSV."
)
print("Verificado: ninguna otra columna presenta valores faltantes")


Valores faltantes por columna:
TotalCharges    11
dtype: int64

Filas duplicadas: 0
IDs duplicados: 0
Verificado: todos los faltantes tienen tenure = 0
Verificado: ninguna otra columna presenta valores faltantes


# 4. Separación de predictores y objetivo

`Churn` se codifica como 0 (permanece) o 1 (abandona); `customerID` y `Churn` no se incluyen como predictores.

In [4]:
# Variable objetivo: 1 = abandono, 0 = permanencia
y = df["Churn"].map({"No": 0, "Yes": 1})

# Variables predictoras
X = df.drop(columns=["customerID", "Churn"])

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nDistribución de clases:")
print(y.value_counts())


# Verificar que Churn se haya convertido correctamente
assert y.notna().all(), (
    "La variable objetivo contiene valores no reconocidos"
)

print("Variable objetivo codificada correctamente")


Dimensiones de X: (7043, 19)
Dimensiones de y: (7043,)

Distribución de clases:
Churn
0    5174
1    1869
Name: count, dtype: int64
Variable objetivo codificada correctamente


# 5. Identificación de variables

Se distinguen 3 características cuantitativas y 16 categóricas. `TotalCharges` se procesa por separado porque es la única columna con faltantes.

In [5]:
# Variables numéricas generales
NUMERIC_FEATURES = (
    "tenure",
    "MonthlyCharges"
)

# TotalCharges se trata por separado porque
# tiene valores faltantes que imputaremos con cero
TOTALCHARGES_FEATURE = (
    "TotalCharges",
)

# Variables categóricas
CATEGORICAL_FEATURES = tuple(
    col for col in X.columns
    if col not in NUMERIC_FEATURES + TOTALCHARGES_FEATURE
)

# Lista completa de características
ALL_FEATURES = (
    NUMERIC_FEATURES
    + TOTALCHARGES_FEATURE
    + CATEGORICAL_FEATURES
)

# Comprobar que no falte ni se repita ninguna variable
assert set(ALL_FEATURES) == set(X.columns)
assert len(ALL_FEATURES) == len(set(ALL_FEATURES))
assert len(ALL_FEATURES) == 19
assert len(CATEGORICAL_FEATURES) == 16

print("Variables numéricas:",
      len(NUMERIC_FEATURES) + len(TOTALCHARGES_FEATURE))

print("Variables categóricas:",
      len(CATEGORICAL_FEATURES))

print("Total de características:",
      len(ALL_FEATURES))

print("\nNuméricas:", NUMERIC_FEATURES + TOTALCHARGES_FEATURE)
print("\nCategóricas:", CATEGORICAL_FEATURES)



Variables numéricas: 3
Variables categóricas: 16
Total de características: 19

Numéricas: ('tenure', 'MonthlyCharges', 'TotalCharges')

Categóricas: ('gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod')


# 6. División Train / Validation / Test

Partición estratificada 60 % / 20 % / 20 %, con `random_state=42`. Los ajustes de preprocesamiento se hacen **solo con Train**. Esta partición deberá ser idéntica para todo el grupo.

In [6]:
from sklearn.model_selection import train_test_split

# Primera división: 80% desarrollo y 20% prueba
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Segunda división: 60% entrenamiento y 20% validación
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDistribución de clases:")
print("Train:", y_train.value_counts(normalize=True))
print("Val:", y_val.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))



Train: (4225, 19)
Validation: (1409, 19)
Test: (1409, 19)

Distribución de clases:
Train: Churn
0    0.734675
1    0.265325
Name: proportion, dtype: float64
Val: Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64
Test: Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


# 7. Inspección de valores faltantes

Se confirma la separación de conjuntos y se observa cuántos `TotalCharges` faltan en cada uno. La mediana de Train se muestra como **referencia comparativa**, pero **no se utiliza para imputar**.

In [7]:
# Comprobar que los conjuntos sean independientes
assert X_train.index.intersection(X_val.index).empty
assert X_train.index.intersection(X_test.index).empty
assert X_val.index.intersection(X_test.index).empty

print("No hay clientes compartidos entre conjuntos.")

# Revisar valores faltantes
print("\nFaltantes en Train:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nFaltantes en Validation:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("\nFaltantes en Test:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

# Inspeccionar clientes con TotalCharges faltante
missing_train = X_train[X_train["TotalCharges"].isna()]

display(
    missing_train[
        ["tenure", "MonthlyCharges", "TotalCharges", "Contract"]
    ]
)

print("Mediana de TotalCharges (Train):")
print(X_train["TotalCharges"].median())


No hay clientes compartidos entre conjuntos.

Faltantes en Train:
TotalCharges    7
dtype: int64

Faltantes en Validation:
TotalCharges    1
dtype: int64

Faltantes en Test:
TotalCharges    3
dtype: int64


,tenure,MonthlyCharges,TotalCharges,Contract
6670,0,73.35,NaN,Two year
3826,0,25.35,NaN,Two year
3331,0,19.85,NaN,Two year
1082,0,25.75,NaN,Two year
1340,0,56.05,NaN,Two year
4380,0,20.00,NaN,Two year
6754,0,61.90,NaN,Two year


Mediana de TotalCharges (Train):
1385.3


# 7.1. Análisis de correlación y redundancia

Las correlaciones se calculan solo con Train. El producto `tenure × MonthlyCharges` sirve para analizar la relación con `TotalCharges`: **no se añade como variable predictora**. Una correlación elevada no obliga a eliminar características.

In [8]:
# Analizar relaciones entre variables numéricas
# utilizando exclusivamente el conjunto Train

num_train = X_train[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].copy()

# Para este análisis utilizamos la misma regla
# de imputación propuesta para TotalCharges
num_train["TotalCharges"] = (
    num_train["TotalCharges"].fillna(0)
)

# Matriz de correlaciones de Pearson
correlaciones = num_train.corr()

print("Correlaciones entre variables numéricas:")
display(correlaciones.round(3))

# Relación entre cargos totales y el producto
# de antigüedad por cargo mensual
aproximacion = (
    num_train["tenure"]
    * num_train["MonthlyCharges"]
)

r = num_train["TotalCharges"].corr(aproximacion)

print(
    "\nCorrelación entre TotalCharges y "
    "tenure × MonthlyCharges:",
    round(r, 4)
)


Correlaciones entre variables numéricas:


,tenure,MonthlyCharges,TotalCharges
tenure,1.000,0.259,0.834
MonthlyCharges,0.259,1.000,0.650
TotalCharges,0.834,0.650,1.000



Correlación entre TotalCharges y tenure × MonthlyCharges: 0.9996


# 8. Definir transformaciones

**Única imputación efectiva:** los faltantes de `TotalCharges` se sustituyen por cero. No se usan imputadores por mediana ni por moda, porque las demás variables están completas en este CSV. Se aplica One-Hot Encoding a las categóricas y StandardScaler únicamente a las numéricas para regresión logística. Los preprocesadores se crean sin ajustar.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


def make_preprocessor(model_type):
    """Construye un preprocesador sin ajustar para 'logistic' o 'tree'."""
    if model_type not in ("logistic", "tree"):
        raise ValueError("Use 'logistic' o 'tree'")

    # tenure y MonthlyCharges están completos: no se imputan.
    # Se escalan solo cuando el modelo es regresión logística.
    numeric_transformer = (
        StandardScaler() if model_type == "logistic" else "passthrough"
    )

    # TotalCharges es la única variable que necesita imputación.
    totalcharges_steps = [
        ("imputer", SimpleImputer(strategy="constant", fill_value=0))
    ]
    if model_type == "logistic":
        totalcharges_steps.append(("scaler", StandardScaler()))

    # Las categóricas están completas: solamente se codifican.
    categorical_transformer = OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, list(NUMERIC_FEATURES)),
            ("totalcharges", Pipeline(totalcharges_steps), list(TOTALCHARGES_FEATURE)),
            ("cat", categorical_transformer, list(CATEGORICAL_FEATURES)),
        ],
        remainder="drop",
    )


# Ambos objetos se entregan sin ajustar.
preprocessor_lr = make_preprocessor("logistic")
preprocessor_dt = make_preprocessor("tree")
print("Preprocesadores definidos correctamente")


Preprocesadores definidos correctamente


# 9. Prueba técnica de los preprocesadores

Se ajusta una copia de cada preprocesador **exclusivamente con Train**, y se transforma Validation sin volver a ajustar. Se comprueba que hay 46 columnas, que `TotalCharges` aparece una sola vez y que no quedan valores no finitos.

In [10]:
import numpy as np
from sklearn.base import clone

for nombre, preprocessor in [
    ("Regresión logística", preprocessor_lr),
    ("Árbol de decisión", preprocessor_dt)
]:

    # Crear una copia independiente
    prep_check = clone(preprocessor)

    # Ajustar exclusivamente con Train
    X_train_processed = prep_check.fit_transform(X_train)

    # Transformar Validation sin volver a ajustar
    X_val_processed = prep_check.transform(X_val)

    # Obtener nombres de características
    feature_names = list(prep_check.get_feature_names_out())

    # Verificar las dimensiones
    assert X_train_processed.shape[0] == len(X_train)
    assert X_val_processed.shape[0] == len(X_val)

    assert X_train_processed.shape[1] == X_val_processed.shape[1]

    # Verificar ausencia de valores inválidos
    assert np.isfinite(X_train_processed).all()
    assert np.isfinite(X_val_processed).all()

    # Verificar que TotalCharges no esté duplicada
    assert feature_names.count(
        "totalcharges__TotalCharges"
    ) == 1

    assert "num__TotalCharges" not in feature_names

    # Verificar cantidad esperada de columnas
    assert len(feature_names) == 46

    print(f"\nModelo: {nombre}")
    print("Train:", X_train_processed.shape)
    print("Validation:", X_val_processed.shape)
    print("Características finales:", len(feature_names))
    print("TotalCharges no está duplicada: OK")
    print("Sin NaN ni infinitos: OK")




Modelo: Regresión logística
Train: (4225, 46)
Validation: (1409, 46)
Características finales: 46
TotalCharges no está duplicada: OK
Sin NaN ni infinitos: OK

Modelo: Árbol de decisión
Train: (4225, 46)
Validation: (1409, 46)
Características finales: 46
TotalCharges no está duplicada: OK
Sin NaN ni infinitos: OK


# 10. Comprobación específica de la imputación con cero

En el preprocesador del árbol (sin escalamiento), los siete `TotalCharges` faltantes de Train deben aparecer como cero. Esta prueba comprueba la ejecución del código, no demuestra el valor real que tenían los datos ausentes.

In [11]:
from sklearn.base import clone

# Usamos el preprocesador del árbol porque no escala
prep = clone(preprocessor_dt)

X_train_check = prep.fit_transform(X_train)

feature_names = prep.get_feature_names_out()

# Localizar la columna TotalCharges transformada
idx = list(feature_names).index(
    "totalcharges__TotalCharges"
)

# Identificar registros originalmente faltantes
mask = X_train["TotalCharges"].isna().to_numpy()

# Comprobar la imputación
valores_imputados = X_train_check[mask, idx]

print("Valores imputados:", valores_imputados)

assert np.all(valores_imputados == 0)

print("Imputación verificada correctamente.")


Valores imputados: [0. 0. 0. 0. 0. 0. 0.]
Imputación verificada correctamente.
